<a href="https://colab.research.google.com/github/AdelineKwakye/walmart-huricane-project/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DATA CLEANING**

In [2]:
# what needs to be done:

  # handle duplicates

  # convert unit price from text -> real number

  # decide on whether negatives values are kept ( yes because they indicate returns / refunds )

  # parse timestamp to turn it into date time

  # clean store_id


# import data
import pandas as pd
walmart_transactions = 'https://raw.githubusercontent.com/AdelineKwakye/walmart-huricane-project/refs/heads/main/data/walmart_transactions.csv'
store_locations = 'https://raw.githubusercontent.com/AdelineKwakye/walmart-huricane-project/refs/heads/main/data/store_locations.csv'
distribution_centers = 'https://raw.githubusercontent.com/AdelineKwakye/walmart-huricane-project/refs/heads/main/data/distribution_centers.csv'
df_transactions = pd.read_csv(walmart_transactions)
df_store = pd.read_csv(store_locations)
df_distribution_centers = pd.read_csv(distribution_centers)
df_product = pd.read_csv('https://raw.githubusercontent.com/AdelineKwakye/walmart-huricane-project/refs/heads/main/data/product_catalog.csv')


print("Store data\n")
print(df_store.head())
print("transactions sample\n")
print(df_transactions.head(10))
print("distribution centers\n")
display(df_distribution_centers)
print('product catalog\n')
display(df_product)


Store data

  store_id                             store_name          city    state  \
0   FL-239            Walmart Supercenter Orlando       Orlando       FL   
1   FL-105              Walmart Supercenter Tampa         Tampa       FL   
2   FL-330              Walmart Supercenter Miami         Miami       FL   
3   FL-412       Walmart Supercenter Jacksonville  Jacksonville       FL   
4   FL-287  Walmart Neighborhood Market Kissimmee     Kissimmee  Florida   

   zip_code  latitude  longitude   region  
0     32801   28.5383   -81.3792  Central  
1     33602   27.9506   -82.4572     West  
2     33101   25.7617   -80.1918    South  
3     32099   30.3322   -81.6557    North  
4     34741   28.2920   -81.4076  Central  
transactions sample

  transaction_id store_id            timestamp            sku  \
0      TXN012636   FL-710  2024-09-24T09:48:13     PNUTBTR-16   
1      TXN000759   fl-412  2024-09-13 20:30:20      CNSOUP-01   
2      TXN003338   FL-105  2024-09-17 10:06:00  TOI

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0
3,DC-MIA,Miami Distribution Center,Miami,FL,350,~750,235,298.0
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN


product catalog



,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,AA,NaN,3M Company,6 pack
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack


In [3]:
# setting up decision log
# used same formating from decision log in previous lab
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df_transactions)
print('starting with', raw_rows, 'rows')

starting with 25562 rows


In [4]:
# handle duplicate function

def handle_duplicates (df_name):
  before_len = len(df_name) # length before
  print("Length before cleaning duplicates: ", before_len)
  removed = df_name.duplicated().sum() # duplicated rows to be removed
  clean_dup = df_name.drop_duplicates().copy() # drop duplicates
  after_len = len(clean_dup) # length after
  print( "Length after cleaning duplicates: ", after_len)
  log('duplicates', 'dropped exact duplicate rows', removed) # log
  return clean_dup

df_clean = handle_duplicates(df_transactions)
show_log()

Length before cleaning duplicates:  25562
Length after cleaning duplicates:  25532
[duplicates] dropped exact duplicate rows (30 row(s))


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30


In [5]:
# handle unit price
def clean_unit_price(df_name):
    before_len = len(df_name)
    print("Length before cleaning: ", before_len)
    clean_df = df_name.copy()


    # Strip '$' and convert directly to float
    clean_df['unit_price'] = (
        clean_df['unit_price']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.strip()
        .astype(float)
    )

    assert clean_df['unit_price'].dtype == float # confirm that it's a float

    after_len = len(clean_df)
    print("Length after cleaning: ", after_len)
    log('unit price', 'stripped $ and converted to float', len(clean_df))
    return clean_df

df_clean = clean_unit_price(df_clean)
show_log()

Length before cleaning:  25532
Length after cleaning:  25532
[unit price] stripped $ and converted to float (25532 row(s))


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532


In [6]:
# handle negative values
def handle_negative_values(df_name):
  # keeping negative values since representative of refunds/ returns
  # this function will just ensure that quantity is a numeric value

  before_len = len(df_name)
  print("Length before cleaning: ", before_len)
  clean_df = df_name.copy()

  clean_df['quantity'] = pd.to_numeric(clean_df['quantity'], errors="coerce") # convert to numeric

  # keep track of negative values
  negative_values = (clean_df['quantity'] < 0 ).sum()
  print("Number of negative values: ", negative_values)

  after_len = len(clean_df)
  print("Length after cleaning: ", after_len)

  log('negative quantity', 'decided to keep all of the negative quantities since it could be indicative of refunds/returns', 0)
  return clean_df

df_clean = handle_negative_values(df_clean)
show_log()

Length before cleaning:  25532
Number of negative values:  130
Length after cleaning:  25532
[negative quantity] decided to keep all of the negative quantities since it could be indicative of refunds/returns (0 row(s))


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0


In [7]:
# cleaning store_id
def clean_store_id(df_name, table_name):
    before_len = len(df_name)
    print(f"Length before cleaning {table_name}: ", before_len)
    clean_df = df_name.copy()

    # standardize the store id to start with FL-###
    # using format that was done in previous lab/studio
    clean_df['store_id'] = (
        clean_df['store_id']
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r'[^A-Z0-9]', '', regex=True)
        .apply(lambda s: f'{s[:2]}-{s[2:]}')
    )

    after_len = len(clean_df)
    print(f"Length after cleaning {table_name}: ", after_len)

    log(
        f'store_id ({table_name})',
        f'standardized formatting of store_id to follow FL-### in {table_name}',
        len(clean_df),
    )
    return clean_df

df_clean = clean_store_id(df_clean, table_name='transactions') # standardize store id from transactions
df_clean_store = clean_store_id(df_store, table_name='stores') # standardize store id from stores
show_log()

Length before cleaning transactions:  25532
Length after cleaning transactions:  25532
[store_id (transactions)] standardized formatting of store_id to follow FL-### in transactions (25532 row(s))
Length before cleaning stores:  14
Length after cleaning stores:  14
[store_id (stores)] standardized formatting of store_id to follow FL-### in stores (14 row(s))


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14


In [8]:
# view list of clean store names
print("df_clean store_id samples:\n", df_clean['store_id'].head(3))
print("df_clean_store store_id samples:\n", df_clean_store['store_id'].head(3))

# confirm that the store names match
print(df_clean['store_id'].isin(df_clean_store['store_id']).all())
print(df_clean_store['store_id'].isin(df_clean['store_id']).all())

# view the updated decisions log
show_log()

df_clean store_id samples:
 0    FL-710
1    FL-412
2    FL-105
Name: store_id, dtype: object
df_clean_store store_id samples:
 0    FL-239
1    FL-105
2    FL-330
Name: store_id, dtype: object
True
True


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14


In [9]:
def clean_timestamps(df_name):
  before_len = len(df_name)
  print("Length before cleaning: ", before_len)
  clean_df = df_name.copy()

  # standardize the timestamp column so it uses space as a separator and a T

  clean_df['timestamp'] = (
      clean_df['timestamp']
      .astype(str)
      .str.strip()
      .str.replace('T', ' ', regex=False)
  )
  # convert to datetime

  clean_df['timestamp'] = pd.to_datetime(clean_df['timestamp'], format='mixed', errors='coerce')

  null_count = clean_df['timestamp'].isnull().sum()
  print("Number of null values: ", null_count)
  # drop na
  clean_df = clean_df.dropna(subset=['timestamp']).copy()

  # create normalized date column
  clean_df['date'] = clean_df['timestamp'].dt.normalize()

  after_len = len(clean_df)
  print("Length after cleaning, ", after_len)
  log('timestamp', 'standardized the separator, converted to be datetime, droped na values, and created a normalized date column ', null_count)
  return clean_df

print('nulls in df_clean: ', df_clean['timestamp'].isnull().sum())

df_clean = clean_timestamps(df_clean)
print('nulls in df_clean: ', df_clean['timestamp'].isnull().sum())
show_log()



# Check remaining nulls across all columns
print("Remaining nulls per column:")
print(df_clean.isnull().sum())

nulls in df_clean:  884
Length before cleaning:  25532
Number of null values:  884
Length after cleaning,  24648
[timestamp] standardized the separator, converted to be datetime, droped na values, and created a normalized date column  (884 row(s))
nulls in df_clean:  0
Remaining nulls per column:
transaction_id    0
store_id          0
timestamp         0
sku               0
product_name      0
category          0
quantity          0
unit_price        0
date              0
dtype: int64


In [10]:
def profile(df, name='dataset'):
    print(f'=== {name} ===')
    print(f'{len(df):,} rows x {len(df.columns)} columns')
    print()
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isnull().sum(),
        'null_pct': (100 * df.isnull().mean()).round(1),
        'distinct': df.nunique(),
    })
    print(summary)
    print()
    print('exact duplicate rows:', df.duplicated().sum())
    return summary


    # Final post-cleaning health check
profile(df_clean, name='Cleaned Transactions (df_clean)')

=== Cleaned Transactions (df_clean) ===
24,648 rows x 9 columns

                         dtype  nulls  null_pct  distinct
transaction_id          object      0       0.0     24648
store_id                object      0       0.0        12
timestamp       datetime64[ns]      0       0.0     23417
sku                     object      0       0.0        45
product_name            object      0       0.0        63
category                object      0       0.0        37
quantity                 int64      0       0.0        11
unit_price             float64      0       0.0      1667
date            datetime64[ns]      0       0.0        30

exact duplicate rows: 0


,dtype,nulls,null_pct,distinct
transaction_id,object,0,0.0,24648
store_id,object,0,0.0,12
timestamp,datetime64[ns],0,0.0,23417
sku,object,0,0.0,45
product_name,object,0,0.0,63
category,object,0,0.0,37
quantity,int64,0,0.0,11
unit_price,float64,0,0.0,1667
date,datetime64[ns],0,0.0,30


In [11]:
# clean distribution center csv
  # unnecessary ~ in surge_capacity column
  # Nan value in cost per shipment


print(df_distribution_centers.info())
print(df_distribution_centers.isnull().sum())

def clean_distribution_center(df_name):
  before_len = len(df_name)
  print("Length before cleaning: ", before_len)
  clean_df = df_name.copy()
  clean_df['surge_capacity'] = clean_df['surge_capacity'].str.strip().str.replace('~', '', regex=False)
  clean_df['surge_capacity'] = pd.to_numeric(clean_df['surge_capacity'], errors='coerce')

  # Count missing shipping costs
  missing_costs = clean_df['cost_per_shipment'].isna().sum()
  print("Missing shipping costs:", missing_costs)

  # Check for missing shipping costs
  missing_costs = clean_df['cost_per_shipment'].isna().sum()
  print("Missing shipping costs:", missing_costs)

  after_len = len(clean_df)
  ## log
  log('surge capacity', 'striped white spaces, got rid of ~, and converted type to be numeric', after_len)
  log('distribution center missing values', 'kept missing value because there is no reliable information to determine correct value', 0)

  return clean_df

clean_distribution_center_df = clean_distribution_center(df_distribution_centers)
show_log()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dc_id                   5 non-null      object 
 1   dc_name                 5 non-null      object 
 2   city                    5 non-null      object 
 3   state                   5 non-null      object 
 4   avg_daily_shipments     5 non-null      int64  
 5   surge_capacity          5 non-null      object 
 6   distance_to_orlando_mi  5 non-null      int64  
 7   cost_per_shipment       4 non-null      float64
dtypes: float64(1), int64(2), object(5)
memory usage: 452.0+ bytes
None
dc_id                     0
dc_name                   0
city                      0
state                     0
avg_daily_shipments       0
surge_capacity            0
distance_to_orlando_mi    0
cost_per_shipment         1
dtype: int64
Length before cleaning:  5
Missing shipping costs: 1
Missing 

,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14
5,timestamp,"standardized the separator, converted to be da...",884
6,surge capacity,"striped white spaces, got rid of ~, and conver...",5
7,distribution center missing values,kept missing value because there is no reliabl...,0


In [23]:
#strip product name
df_clean['product_name']=df_clean['product_name'].str.strip()
#lowercase
df_clean['product_name']=df_clean['product_name'].str.lower()
#remove spaces
df_clean['product_name']=df_clean['product_name'].str.replace(' ','')

mapping_rules = {
    'batteries': 'BAT-AA4',
    'formula': 'BFORM-01',
    'beer': 'BEER-24',
    'monopoly': 'MONOPOLY-01',
    'whitebread': 'BREAD-WH',
    'candle': 'CANDLE-EMRG',
    'playingcards': 'CARDS-01',
    'variety': 'CHIPS-VP',
    'charger': 'CHRG-PORT',
    'diaper': 'DIAPERS-3',
    'firstaid': 'FAID-01',
    'aidkit': 'FAID-01',
    'flashlight': 'FLASH-01',
    'tarp': 'HDTARP-01',
    'papertowel': 'PTOWEL-6',
    'peanut': 'PNUTBTR-16',
    'poptart': 'POPTART-STRAW',
    'strawberrypop': 'POPTART-STRAW',
    'pop-tartsstrawberry': 'POPTART-STRAW',
    'streaming': 'ROKU-STD',
    'roku': 'ROKU-STD',
    'soup': 'CNSOUP-01',
    'toiletpaper': 'TP-12',
    'water': 'WTR-24'
}

# Apply mapping rules
for key, sku in mapping_rules.items():
    df_clean.loc[df_clean['product_name'].str.contains(key), 'sku'] = sku

# Show finalized groups to verify all product names mapped to single SKUs
print(df_clean.groupby('sku')['product_name'].unique())

# Map incorrect SKUs to the correct consolidated SKUs
sku_remap = {
    'WHITEBREAD': 'BREAD-WH',
    'TOILETPAPER12': 'TP-12'
}

df_clean['sku'] = df_clean['sku'].replace(sku_remap)

# Re-apply the standardization logic to clean up the product names for these updated SKUs
canonical_names = {
    'BREAD-WH': 'whitebread',
    'TP-12': 'toiletpaper12pk'
}

for sku, name in canonical_names.items():
    df_clean.loc[df_clean['sku'] == sku, 'product_name'] = name

#log mapping decision
log(
    'sku mapping',
    'mapped and standardized variations of product names into standardized SKUs based on pattern matching rules',
    len(df_clean)
)

# view the updated decisions log
show_log()

sku
BAT-AA4              [aabatteries4-pack]
BEER-24                  [lightbeer24pk]
BFORM-01                   [babyformula]
BREAD-WH                    [whitebread]
CANDLE-EMRG          [candles-emergency]
CARDS-01                  [playingcards]
CHIPS-VP              [chipsvarietypack]
CHRG-PORT        [phonecharger-portable]
CNSOUP-01                   [cannedsoup]
DIAPERS-3                 [diaperssize3]
FAID-01                    [firstaidkit]
FLASH-01                 [ledflashlight]
HDTARP-01                [heavydutytarp]
MONOPOLY-01         [boardgame-monopoly]
PNUTBTR-16                [peanutbutter]
POPTART-STRAW      [pop-tartsstrawberry]
PTOWEL-6             [papertowels6-roll]
ROKU-STD               [streamingdevice]
TP-12                  [toiletpaper12pk]
WTR-24             [bottledwater24-pack]
Name: product_name, dtype: object
[sku mapping] mapped and standardized variations of product names into standardized SKUs based on pattern matching rules (24648 row(s))


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14
5,timestamp,"standardized the separator, converted to be da...",884
6,surge capacity,"striped white spaces, got rid of ~, and conver...",5
7,distribution center missing values,kept missing value because there is no reliabl...,0
8,sku mapping,mapped and standardized variations of product ...,24648
9,product name standardization,rewrote variations of product names to a singl...,24648


In [24]:
def standardize_product_names(df):
    clean_df = df.copy()

    # Group by SKU and pick the most common (mode) or first cleaned product name as the canonical representative
    canonical_names = (
        clean_df.groupby('sku')['product_name']
        .agg(lambda x: x.value_counts().idxmax() if not x.empty else None)
        .to_dict()
    )

    # Apply the mapping to overwrite individual variations
    clean_df['product_name'] = clean_df['sku'].map(canonical_names).fillna(clean_df['product_name'])

    # Log the decision
    log('product name standardization', 'rewrote variations of product names to a single canonical name per SKU', len(clean_df))
    return clean_df

df_clean = standardize_product_names(df_clean)

# Verify that each SKU now maps to exactly one unique product name
print(df_clean.groupby('sku')['product_name'].unique())
show_log()

[product name standardization] rewrote variations of product names to a single canonical name per SKU (24648 row(s))
sku
BAT-AA4              [aabatteries4-pack]
BEER-24                  [lightbeer24pk]
BFORM-01                   [babyformula]
BREAD-WH                    [whitebread]
CANDLE-EMRG          [candles-emergency]
CARDS-01                  [playingcards]
CHIPS-VP              [chipsvarietypack]
CHRG-PORT        [phonecharger-portable]
CNSOUP-01                   [cannedsoup]
DIAPERS-3                 [diaperssize3]
FAID-01                    [firstaidkit]
FLASH-01                 [ledflashlight]
HDTARP-01                [heavydutytarp]
MONOPOLY-01         [boardgame-monopoly]
PNUTBTR-16                [peanutbutter]
POPTART-STRAW      [pop-tartsstrawberry]
PTOWEL-6             [papertowels6-roll]
ROKU-STD               [streamingdevice]
TP-12                  [toiletpaper12pk]
WTR-24             [bottledwater24-pack]
Name: product_name, dtype: object


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14
5,timestamp,"standardized the separator, converted to be da...",884
6,surge capacity,"striped white spaces, got rid of ~, and conver...",5
7,distribution center missing values,kept missing value because there is no reliabl...,0
8,sku mapping,mapped and standardized variations of product ...,24648
9,product name standardization,rewrote variations of product names to a singl...,24648


In [22]:
# Map incorrect SKUs to the correct consolidated SKUs
sku_remap = {
    'WHITEBREAD': 'BREAD-WH',
    'TOILETPAPER12': 'TP-12'
}

df_clean['sku'] = df_clean['sku'].replace(sku_remap)

# Re-apply the standardization logic to clean up the product names for these updated SKUs
canonical_names = {
    'BREAD-WH': 'whitebread',
    'TP-12': 'toiletpaper12pk'
}

for sku, name in canonical_names.items():
    df_clean.loc[df_clean['sku'] == sku, 'product_name'] = name



[sku consolidation] consolidated duplicate SKUs for bread and toilet paper (2243 row(s))
sku
BAT-AA4              [aabatteries4-pack]
BEER-24                  [lightbeer24pk]
BFORM-01                   [babyformula]
BREAD-WH                    [whitebread]
CANDLE-EMRG          [candles-emergency]
CARDS-01                  [playingcards]
CHIPS-VP              [chipsvarietypack]
CHRG-PORT        [phonecharger-portable]
CNSOUP-01                   [cannedsoup]
DIAPERS-3                 [diaperssize3]
FAID-01                    [firstaidkit]
FLASH-01                 [ledflashlight]
HDTARP-01                [heavydutytarp]
MONOPOLY-01         [boardgame-monopoly]
PNUTBTR-16                [peanutbutter]
POPTART-STRAW      [pop-tartsstrawberry]
PTOWEL-6             [papertowels6-roll]
ROKU-STD               [streamingdevice]
TP-12                  [toiletpaper12pk]
WTR-24             [bottledwater24-pack]
Name: product_name, dtype: object


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14
5,timestamp,"standardized the separator, converted to be da...",884
6,surge capacity,"striped white spaces, got rid of ~, and conver...",5
7,distribution center missing values,kept missing value because there is no reliabl...,0
8,sku mapping,mapped and standardized variations of product ...,24648
9,product name standardization,rewrote variations of product names to a singl...,24648


In [30]:
#clean category
df_clean['category']=df_clean['category'].str.strip()
df_clean['category']=df_clean['category'].str.lower()
df_clean['category']=df_clean['category'].str.replace(' ','')
df_clean['category']=df_clean['category'].str.replace('-','')

# define remapping dictionary to consolidate duplicates
category_map = {
    'groceries': 'grocery',
    'household': 'household_essentials',
    'householdessentials': 'household_essentials',
    'alcohol': 'alcoholic_beverages',
    'alcoholicbeverages': 'alcoholic_beverages',
    'emergencysupplies': 'emergency',
    'baby': 'baby_products',
    'babyproducts': 'baby_products',
    'games&entertainment': 'entertainment',
    'breakfastfoods': 'breakfast',
    'health&wellness': 'health'
}

# Apply the category remapping
df_clean['category'] = df_clean['category'].replace(category_map)

# Log the decision
log(
    'category consolidation',
    'consolidated overlapping and duplicate category names to standard categories',
    len(df_clean)
)

# list unique categories to verify no duplicates remain
print("Standardized Categories:", df_clean['category'].unique())
show_log()

[category consolidation] consolidated overlapping and duplicate category names to standard categories (24648 row(s))
Standardized Categories: ['grocery' 'household_essentials' 'electronics' 'alcoholic_beverages'
 'snacks' 'emergency' 'entertainment' 'health' 'breakfast' 'baby_products']


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14
5,timestamp,"standardized the separator, converted to be da...",884
6,surge capacity,"striped white spaces, got rid of ~, and conver...",5
7,distribution center missing values,kept missing value because there is no reliabl...,0
8,sku mapping,mapped and standardized variations of product ...,24648
9,product name standardization,rewrote variations of product names to a singl...,24648


In [31]:
category_remap_extra = {
    'snacks': 'food',
    'electronics': 'entertainment'
}

df_clean['category'] = df_clean['category'].replace(category_remap_extra)

log(
    'category consolidation extra',
    "mapped 'snacks' to 'food' and 'electronics' to 'entertainment'",
    len(df_clean)
)

print("Updated Standardized Categories:", df_clean['category'].unique())
show_log()

[category consolidation extra] mapped 'snacks' to 'food' and 'electronics' to 'entertainment' (24648 row(s))
Updated Standardized Categories: ['grocery' 'household_essentials' 'entertainment' 'alcoholic_beverages'
 'food' 'emergency' 'health' 'breakfast' 'baby_products']


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,unit price,stripped $ and converted to float,25532
2,negative quantity,decided to keep all of the negative quantities...,0
3,store_id (transactions),standardized formatting of store_id to follow ...,25532
4,store_id (stores),standardized formatting of store_id to follow ...,14
5,timestamp,"standardized the separator, converted to be da...",884
6,surge capacity,"striped white spaces, got rid of ~, and conver...",5
7,distribution center missing values,kept missing value because there is no reliabl...,0
8,sku mapping,mapped and standardized variations of product ...,24648
9,product name standardization,rewrote variations of product names to a singl...,24648


In [32]:
df_clean

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,date
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,peanutbutter,grocery,1,3.15,2024-09-24
1,TXN000759,FL-412,2024-09-13 20:30:20,CNSOUP-01,cannedsoup,grocery,1,2.02,2024-09-13
2,TXN003338,FL-105,2024-09-17 10:06:00,TP-12,toiletpaper12pk,household_essentials,2,10.47,2024-09-17
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,streamingdevice,entertainment,1,28.59,2024-09-11
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,lightbeer24pk,alcoholic_beverages,3,18.82,2024-09-26
...,...,...,...,...,...,...,...,...,...
25557,TXN020201,FL-105,2024-09-03 18:28:07,DIAPERS-3,diaperssize3,baby_products,2,24.32,2024-09-03
25558,TXN014835,FL-239,2024-09-26 08:47:08,CARDS-01,playingcards,entertainment,3,5.00,2024-09-26
25559,TXN002434,FL-239,2024-09-16 16:39:00,PTOWEL-6,papertowels6-roll,household_essentials,2,7.78,2024-09-16
25560,TXN014739,FL-710,2024-09-25 17:08:35,CHRG-PORT,phonecharger-portable,entertainment,1,24.18,2024-09-25
